In [ ]:
import os
import json
from collections import defaultdict

import gensim.downloader as api
from gensim.models import Word2Vec
import networkx as nx
from tqdm import tqdm
from nltk.corpus import wordnet as wn
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

### Evaluation

In [ ]:
# Read in graphs
graph_weighted = nx.read_gexf('data/graph_weighted_with_probs.gexf')
graph_original = nx.read_gexf('data/pos_tagged_graph_original.gexf')

In [ ]:
# Fetch Google Pre-trained models
models = {}
models['google_word2vec'] = api.load('word2vec-google-news-300')
models['weighted_window_5'] = Word2Vec.load("data/weighted_graph_embeddings_window_5.model").wv
models['weighted_window_10'] = Word2Vec.load("data/weighted_graph_embeddings_window_10.model").wv
models['unweighted_window_5'] = Word2Vec.load("data/unweighted_graph_embeddings_window_5.model").wv
models['unweighted_window_10'] = Word2Vec.load("data/unweighted_graph_embeddings_window_10.model").wv

graph_models = [
    'unweighted_window_5', 
    'unweighted_window_10', 
    'weighted_window_5', 
    'weighted_window_10'
    ]

google_models = ['google_word2vec']

In [ ]:
def get_nodes(graph, word, *pos_info):
    nodes = [n for n, d in graph.nodes(data=True) if 'label' in d and f"'{word.lower()}'" in d['label']]
    if len(pos_info) == 0:
        return nodes

    valid_nodes = []
    for node in nodes:
        match = True
        for item in pos_info:
            if f"'{item}'" not in node:
                match=False
                break
        if match:
            valid_nodes.append(node)

    return valid_nodes

#### Analogy Tests

In [ ]:

# function to calculate accuracies using analogies for the different models
def calculate_analogy_accuracy(analogies_list, model_keys, accuracies):
    for category, analogies in analogies_list.items():
        for analogy in tqdm(analogies, f'calculating for {category}...'):
            w1, r1, w2, r2 = analogy
            
            for model_key in model_keys:
                model = models[model_key]
                
                try:
                    test_vect = model[w1] - model[r1] + model[w2]
                except KeyError:
                    print(model_key, w1, r1, w2, r2)
                    continue
                most_similar = [item[0] for item in model.similar_by_vector(test_vect, topn=10)]
    
                if r2 in most_similar[:1]:
                    accuracies[model_key][category]['top1'] += 1
                    accuracies[model_key][category]['top5'] += 1
                    accuracies[model_key][category]['top10'] += 1
                elif r2 in most_similar[:5]:
                    accuracies[model_key][category]['top5'] += 1
                    accuracies[model_key][category]['top10'] += 1
                elif r2 in most_similar[:10]:
                    accuracies[model_key][category]['top10'] += 1


##### Prep

In [ ]:
# read in anaogies
analogies_folder = 'analogies'
analogies_master = {}
for analogy_file in os.listdir(analogies_folder):
    category = analogy_file[:-4]
    analogies_master[category] = pd.read_table(os.path.join(analogies_folder, analogy_file), sep=" ", header=None)
    analogies_master[category].columns = ['w1', 'r1', 'w2', 'r2']
    

In [ ]:
# get and save a list of all nodes for words in the analogies
word_to_nodes = {}
words_not_in_graph = []
for category, tests in analogies_master.items():
    for ind, row in tqdm(tests.iterrows(), category, total=len(tests)):
        for word in row:
            if word not in word_to_nodes:
                nodes = [node for node in get_nodes(graph_weighted, word) if "'UNK_POS'" not in node]
                if len(nodes) == 0:
                    words_not_in_graph.append(word)
                word_to_nodes[word] = nodes


In [ ]:
# clean analogies
for category, tests in analogies_master.items():
    analogies_master[category] = tests[~tests.isin(words_not_in_graph).any(axis=1)]

In [ ]:
# save a list of nodes associated with each word in the tests
with open('data/word_to_nodes.json', 'w') as json_file:
    json.dump(word_to_nodes, json_file)

##### Tests

In [ ]:
with open('data/word_to_nodes.json', 'r') as json_file:
    word_to_nodes = json.load(json_file)

In [ ]:
google_words = set([word.lower() for word in models['google_word2vec'].index_to_key])

In [ ]:
# create lists of analogies, but in a format of the embeddings models can take. 
# for graphical models, this generates analogies using nodes.
# for google, this generates analogies using words themselves.
# filters out analogies that have words that are either not in the graph or not in google's list
analogy_nodes = defaultdict(list)
analogy_google = defaultdict(list)
for category, tests in analogies_master.items():
    for ind, row in tqdm(tests.iterrows(), category, total=len(tests)):
        w1, r1, w2, r2 = row
        w1_nodes = word_to_nodes[w1]
        r1_nodes = word_to_nodes[r1]
        w2_nodes = word_to_nodes[w2]
        r2_nodes = word_to_nodes[r2]

        nodes1_3 = set()
        for node1 in w1_nodes:     # for each node that is the first word in the analogy
            try:
                node3 = [node for node in w2_nodes if eval(node)[1:] == eval(node1)[1:]][0]    # find the node for the third word that has the same pos
                nodes1_3.add((node1, node3))
            except IndexError:
                continue
            
            
        nodes2_4 = set() 
        for node2 in r1_nodes:    # then for each word that is the second word in the analogy
            try:
                node4 = [node for node in r2_nodes if eval(node)[1:] == eval(node2)[1:]][0]    # find the node for the third word that has the same pos
                nodes2_4.add((node2, node4))
            except IndexError:
                continue

        for node1, node3 in nodes1_3:
            for node2, node4 in nodes2_4:
                if (eval(node1)[0] in google_words) and (eval(node2)[0] in google_words) \
                        and (eval(node3)[0] in google_words) and (eval(node4)[0] in google_words):
                    
                    analogy_nodes[category].append((node1, node2, node3, node4))
                    analogy_google[category].append((w1, r1, w2, r2))
                    
                    
            

In [ ]:
# initialize
accuracies = {}
for model in models:
    accuracies[model] = {}
    for category in analogy_nodes:
        accuracies[model][category] = {}
        accuracies[model][category]['top1'] = 0
        accuracies[model][category]['top5'] = 0
        accuracies[model][category]['top10'] = 0

# calculate accuracies for the four different models
print('Graph models:')
calculate_analogy_accuracy(analogy_nodes, graph_models, accuracies)
print('Google model:')
calculate_analogy_accuracy(analogy_google, google_models, accuracies)


# normalize accuracies
for model in models:
    for category in analogy_nodes:
        num = len(analogy_nodes[category])
        for count_key in accuracies[model][category]:
            accuracies[model][category][count_key] /= num     

In [ ]:
for model_key in models:
    print(model_key)
    display(pd.DataFrame(accuracies[model_key]))


#### Bias Check

**NOTE:** Rough List Generated from ChatGPT. 

In [ ]:
men_occupations = [
    'Engineer',
    'Construction',
    'Firefighter',
    'Mechanic',
    'Plumber',
    'Electrician',
    'Pilot',
    'Information_Technology',
    'Soldier',
    'CEO',
    'Computer_Programmer'
]

women_occupations = [
    'Nurse',
    'Teacher',
    'Secretary',
    'Receptionist',
    'Nanny',
    'Librarian',
    'Waitress',
    'Hairdresser',
    'Flight_attendant',
    'Dancer',
    'Homemaker'
]

In [ ]:
sports_men = [
    'Football',
    'Soccer',
    'Baseball',
    'Basketball',
    'Boxing',
    'Rugby',
    'Wrestling',
    'Ice_Hockey',
    'Golf',
]

sports_women = [
    'Gymnastics',
    'Figure_Skating',
    'Swimming',
    'Ballet',
    'Cheerleading',
    'Equestrian',
    'Netball',
    'Volleyball'
]

In [ ]:
instruments_men = [
    'Electric_Guitar',
    'Drums',
    'Trumpet',
    'Saxophone',
    'Bass_Guitar',
    'Trombone',
    'Violin',
    'Bass',
    'Cello',
]

instruments_women = [
    'Flute',
    'Harp',
    'Clarinet',
    'Viola',
    'Piano',
    'Oboe',
    'Harp',
    'Keyboard',
]

In [ ]:
mens = men_occupations + sports_men + instruments_men
womens = women_occupations + sports_women + instruments_women

In [ ]:
plot_titles = {
    'unweighted': 'Unweighted Graph',
    'weighted_window3': 'Weighted Graph - Node2Vec Window Size 3',
    'weighted_window5': 'Weighted Graph - Node2Vec Window Size 5',
    'google_word2vec': 'Google Word2Vec Model'
}

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 10))
i, j = (0, 0)
for model_key, model in tqdm(models.items()):
    if model_key != 'google_word2vec':  
        man_key = "('man', 'n', 'singular', 'common')"
        woman_key = "('woman', 'n', 'singular', 'common')"
    else:
        man_key = 'man'
        woman_key = 'woman'
        
    man_emb = model[man_key]
    woman_emb = model[woman_key]
    magnitude_man = np.linalg.norm(man_emb)
    magnitude_woman = np.linalg.norm(woman_emb)

    mens_x = []
    mens_y = []

    womens_x = []
    womens_y = []
    
    for item in mens + womens:
        if model_key != 'google_word2vec':  
            nodes = [node for node in get_nodes(graph_weighted, item) if 'UNK_POS' not in node]
        else:
            nodes = [item]

        for node in nodes:
            item_emb = model[node]
            magnitude_item = np.linalg.norm(item_emb)
            
            similarity_man = np.dot(item_emb, man_emb) / (magnitude_man * magnitude_item)
            similarity_woman = np.dot(item_emb, woman_emb) / (magnitude_woman * magnitude_item)
            #ax.annotate(node, (similarity_man, similarity_woman), textcoords="offset points", xytext=(5, 5), ha='left', fontsize=8)
            if item in mens:
                mens_x.append(similarity_man)
                mens_y.append(similarity_woman)
            else:
                womens_x.append(similarity_man)
                womens_y.append(similarity_woman)

    ax = axs[i,j]
    ax.scatter(mens_x, mens_y, color='blue', label='Labeled Men\'s')
    ax.scatter(womens_x, womens_y, color='orange', label='Labeled Women\'s')
    ax.legend()
    ax.set_xlabel('Similarity to Man')
    ax.set_ylabel('Similarity to Woman')
    ax.set_title(plot_titles[model_key])
    ax.set_aspect('equal', adjustable='box')

    j = (j + 1) % 2
    if j == 0:
        i += 1
    
plt.savefig('../figures/plot_of_bias.png')